# LAB-HW-00 — KV260 厂商工具链预检

**今天只解决一个问题：在还没有连接 FPGA 板卡之前，确认 development host 上的 AMD Vivado、JTAG driver 和 KV260 board definition 已经准备好。**

这一步故意不接板、不写 RTL、不生成 bitstream。这样以后 target discovery 失败时，不会把“工具没装好”和“板卡有问题”混在一起。

**Project Trace:** RMD-012 · T-HW-001/T-HW-011

## 1. 今天需要什么

- 一台 **x86-64** development host；
- AMD Vivado 2026.1 安装介质；
- 足够的磁盘空间；
- 网络连接，用于安装器与 Vivado Board Store；
- **今天不需要 KV260 上电，也不需要 microSD。**

### 第一版课程支持的 host OS

为了让零基础步骤可复现，第一版课程把 development-host 教学路径限定在 AMD 2026.1 官方支持的**原生 Windows / Linux**，避免未经验证的环境差异混入上板教学。优先采用：

- Windows 10/11 的 AMD 2026.1 支持版本；
- Ubuntu 22.04 / 24.04 的 AMD 2026.1 支持版本。

macOS、WSL、虚拟机 USB/JTAG passthrough 不作为第一版零基础保证路径；这些环境以后可以作为 porting note，而不能静默替代已测试路径。

### 课程 authoring baseline

本批 Lab 选择 **Vivado 2026.1** 作为 **authoring candidate baseline**，用于让教材、脚本和截图口径一致。它目前还不是“已经实体板验证的 supported baseline”；只有真实 KV260 dry run 留下 evidence 后才能升级为 tested support。真正的 target detection 要到 LAB-HW-02 才能形成 T-HW-002 evidence。

## 2. 安装 Vivado 与 cable driver

安装 AMD Vivado 2026.1 时，确保包含 **Zynq UltraScale+ MPSoC / Kria K26** device support。

安装后先运行：

```bash
vivado -version
```

如果出现 `vivado: command not found`，不要继续 LAB-HW-01/02。先初始化 Vivado 环境或从安装目录启动工具。

### JTAG cable driver

**Windows：** Vivado installer 中选择 **Install Cable Drivers**。如果需要 post-install repair/补装，AMD UG973 给出的 Administrator 命令流程如下（安装目录不同就替换 `%VIVADO_INSTALL_DIR%`）：

```bat
cd %VIVADO_INSTALL_DIR%\data\xicom\cable_drivers\nt64
install_drivers_wrapper.bat %TEMP%
```

**Linux：** AMD 要求安装完成后再以 root/sudo 运行：

```bash
cd <Vivado Install Dir>/data/xicom/cable_drivers/lin64/install_script/install_drivers/
sudo ./install_drivers
```

LAB-HW-00 可以确认“driver 安装步骤已经完成”，但**真正证明 cable/JTAG driver 能工作的是 LAB-HW-02 的真实 target discovery**。不要把“安装脚本成功”写成“板卡已验证”。

> 不要用 OSS CAD Suite 的 Yosys 替代 Vivado。Lesson 13 的 generic synthesis dry run 与 KV260 target-specific implementation/programming 是两层不同工具链。

## 3. 安装/刷新 KV260 board definitions

打开 Vivado：

1. **Tools → Vivado Store**；
2. 进入 **Boards**；
3. Refresh；
4. 搜索 Kria / KV260；
5. 安装 AMD/Xilinx 提供的 KV260/K26 SOM 与 companion-card board data。

AMD 的 KV260 Board Flow 依赖 SOM + carrier/companion-card 元数据。课程脚本不把某个旧教程中的固定 board-part version 写死，而是检查当前安装中是否能找到 `*kv260*` board parts。

## 4. 运行课程 preflight

从仓库根目录运行：

```bash
vivado -mode batch -nolog -nojournal \
  -source boards/kv260/scripts/check_vivado.tcl \
  | tee lab-hw-00-vivado-preflight.txt
```

你要看到两类证据：

- Vivado version；
- 至少一个 KV260 相关 board part。

如果脚本返回非零状态，LAB-HW-00 **没有通过**。

## 5. Expected Evidence

保存：

- `lab-hw-00-vivado-preflight.txt`；
- OS 名称/版本；
- Vivado version；
- 脚本打印的 KV260 board part 列表；
- 当前 Git commit。

不要截图代替文本 log；截图只能作为补充。

### Save Evidence

用 `boards/kv260/evidence/manifest.example.json` 作为 T-HW-011 checklist。复制成 LAB-HW-00 的本地 evidence 文件，按这一次真实操作填写，并列出保留的 log/photo/text artifact。生成在 `boards/kv260/evidence/` 下的 evidence 默认被 Git 忽略；正式验收时再通过 hardware-checkpoint record 发布/关联。

## 6. If it does not work

**A. `vivado` 找不到**  
→ 先修复安装/PATH。不要接板，不要改 RTL。

**B. Vivado 能启动，但脚本显示 `NO_KV260_BOARD_PART`**  
→ 回到 Vivado Store → Boards → Refresh，安装 KV260/K26 board data 后重新运行。

**C. 使用的不是 2026.1**  
→ 记录实际版本并停止继续实体 Lab。课程 baseline 变更需要先更新规范/CI evidence，不能由学生个人静默漂移。

## 7. Human Check

在进入下一 Lab 前，能用自己的话回答：

1. 为什么今天完全不需要 FPGA 板卡？
2. Yosys 能综合 SystemVerilog，为什么它仍不能替代 KV260 的 Vivado implementation/programming flow？
3. board file / board flow 解决了什么问题？
4. 如果明天板卡无法被发现，为什么你已经可以排除一部分“工具未安装”的原因？

## 8. 官方依据

- AMD UG973 2026.1 — Supported Operating Systems  
  https://docs.amd.com/r/en-US/ug973-vivado-release-notes-install-license/Supported-Operating-Systems
- AMD UG973 2026.1 — Install Cable Drivers  
  https://docs.amd.com/r/en-US/ug973-vivado-release-notes-install-license/Install-Cable-Drivers
- AMD UG1089 — KV260 Vivado Board Flow  
  https://docs.amd.com/r/en-US/ug1089-kv260-starter-kit/Vivado-Board-Flow
- AMD UG994 2026.1 — Vivado Store board installation  
  https://docs.amd.com/r/en-US/ug994-vivado-ip-subsystems/Downloading-Board-Files-from-GitHub-Using-the-Vivado